In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import max_error
import pandas as pd
import os
from itertools import compress

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.data import besInferenceDatapoints
from neuro_bes.network import deepNN
from neuro_bes.preprocessing import profile_transform


In [ ]:
tf.config.threading.set_intra_op_parallelism_threads(50)
tf.config.threading.set_inter_op_parallelism_threads(50)

In [ ]:
# tf.config.threading.set_inter_op_parallelism_threads(32)
# tf.config.threading.set_intra_op_parallelism_threads(32)
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
path="/home/molnarbalazs/data/BES_ML_modelling/W7X_op23_SPADE_recons/train_newcuration"
file_list=os.listdir(path)
file_list=[i for i in file_list if "_we" in i]
batch_train=[]
for idx,db_file in enumerate(file_list):
    bes_data=besInferenceDatapoints(path=os.path.join(path,db_file))
    mask=np.max(bes_data.emissions[:,-10:],axis=1)==np.max(bes_data.emissions,axis=1)
    #filter out nan profiles
    mask=np.isnan(bes_data.emissions).any(axis=1) | mask
    mask=np.mean(bes_data.emissions,axis=1)<30 | mask
    bes_data.emissions=bes_data.emissions[~mask]
    bes_data.densities=bes_data.densities[~mask]
    bes_data.tags=list(compress(bes_data.tags,~mask))
    if bes_data.emissions.shape[0]>1:
        batch_train.append(bes_data)

In [ ]:
X_train=[]
Y_train=[]
for batch in batch_train:
    X_train.extend(batch.emissions) 
    Y_train.extend(batch.densities)

In [ ]:
pipeline = profile_transform.PartialInversePipeline([
    ('beam_int_scaler', profile_transform.BeamIntensityScaler()),
    ('density_flatout', profile_transform.DensityFlatout2()),
    ('emission_scaler', profile_transform.EmissionScaler()),
    ('density_scaler', profile_transform.DensityScaler()),
    ('interpolate_to_common_grid', profile_transform.InterpolateToCommonGrid())
])

pipeline_batch_train = pipeline.fit_transform(batch_train)
X_train_scaled=np.concatenate([batch.emissions for batch in pipeline_batch_train])
Y_train_scaled=np.concatenate([batch.densities for batch in pipeline_batch_train])

In [ ]:
X_train_scaled.shape

In [ ]:
r_coord=pipeline['interpolate_to_common_grid'].common_grid_

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(12,6))
for i in (np.random.rand(10)*X_train_scaled.shape[0]).astype(int):
    ax[0,0].plot(X_train[i][::-1])
    ax[0,0].set_title("Original Emission Profiles (Train)")
    ax[0,0].set_xlabel("r (mm)")
    ax[0,0].set_ylabel("Emission")
    ax[0,1].plot(Y_train[i][::-1])
    ax[0,1].set_title("Original Density Profiles (Train)")
    ax[0,1].set_xlabel("r (mm)")
    ax[0,1].set_ylabel("Density")
    ax[1,0].plot(r_coord,X_train_scaled[i])
    ax[1,0].set_title("Scaled Emission Profiles (Train)")
    ax[1,0].set_xlabel("r (mm)")
    ax[1,0].set_ylabel("Scaled Emission")
    ax[1,1].plot(r_coord,Y_train_scaled[i])
    ax[1,1].set_title("Scaled Density Profiles (Train)")
    ax[1,1].set_xlabel("r (mm)")
    ax[1,1].set_ylabel("Scaled Density")
plt.tight_layout()
plt.show()

In [ ]:
line_integrals=[]
for i in pipeline['beam_int_scaler'].integral_emissions_:
    line_integrals.extend(i['area'])
plt.hist(line_integrals, bins=50)

In [ ]:
# shuffle the data
indices=np.arange(X_train_scaled.shape[0])
np.random.shuffle(indices)
train_size=-1
X_train_scaled=X_train_scaled[indices[:train_size]]
Y_train_scaled=Y_train_scaled[indices[:train_size]]

In [ ]:
n_features=r_coord.shape[0]
model=deepNN.make_CNN(data_length=n_features, num_layers=6, filters_per_layer=32, kernel_size_per_layer=15, dropout_rate=0.0, use_batchnorm=False)
#model=deepNN.make_MLP(data_length=n_features, num_layers=6, units_per_layer=128, activations='relu',dropout_rate=0.2)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='mse', metrics=['mape'])
model,history = deepNN.train(model, X_train_scaled, Y_train_scaled, epochs=500, batch_size=64)
model.summary()

In [ ]:
plt.subplot(1, 2, 1)
plt.plot(history.history['val_loss'], label='Validation loss')
plt.plot(history.history['loss'], label='Training loss') 
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation loss')
plt.legend()
plt.grid(True)
plt.ylim(0, history.history['val_loss'][10])  

plt.subplot(1, 2, 2)
plt.plot(history.history['val_mape'], label='Validation MAPE')
plt.plot(history.history['mape'], label='Training MAPE') 
plt.xlabel('Epoch')
plt.ylabel('Mean absolute percentage error')
plt.title('Training and Validation MAPE')
plt.legend()
plt.grid(True)
plt.ylim(0, history.history['val_mape'][10])  
plt.show()

In [ ]:
print('train mse:', np.mean(history.history['loss'][-50:]), 'train mape:', np.mean(history.history['mape'][-50:]), 'val mse:', np.mean(history.history['val_loss'][-50:]), 'val mape:', np.mean(history.history['val_mape'][-50:]))

In [ ]:
import joblib
# save model and pipeline without pickle
model.save("density_prediction_model_newcuration_v2.keras")
#import joblib
joblib.dump(pipeline, "preprocessing_pipeline_newcuration_v2.joblib")  